# Regression Prediction Pipeline (production file)

This notebook reads what the regression work has already produced (the saved models, the frozen blend weight and margins, and the code in `my_val_capstone_01/regression/`) and uses it to **create the production file** in `modeling/`. The file name is set in one cell near the end (`PIPELINE_FILENAME`), so it can be changed before the file is written.

**Nothing is trained, tuned or changed here.** The notebook only *reads* the existing models, tables and code. The only file it writes is the pipeline `.py` in `modeling/`.

**Steps to create the pipeline file**

| Step | What happens |
|---|---|
| 1 | List what already exists (models, package code, `modeling/`) |
| 2 | List the functions that can be reused, read from the `.py` files |
| 3 | Read the saved models, blend weight, margins and the exact 26 feature columns |
| 4 | Design: what goes in, which step does what, what comes out |
| 5 | Choose the file name (change it here if you want another name) |
| 6 | Write the `.py` file into `modeling/` |
| 7 | Check the saved file: same numbers as the saved test predictions, and the raw-items path gives the same result as the training code |

In [1]:
import warnings
warnings.filterwarnings("ignore")
import ast, sys, importlib
from pathlib import Path
import numpy as np, pandas as pd, joblib

ROOT = Path("../..").resolve()                     # project root (this notebook lives in notebooks/regression/)
sys.path.insert(0, str(ROOT))

MODELS = ROOT / "models" / "regression"
INTERIM = ROOT / "data" / "interim" / "regression"
PROCESSED = ROOT / "data" / "processed" / "regression"
PACKAGE_DIR = ROOT / "my_val_capstone_01" / "regression"
MODELING_DIR = ROOT / "modeling"                    # production folder, next to the classification predict.py
print("project root:", ROOT)

project root: D:\UTS MDSI\36127\Project\Client dataset\Notebook\Hosuehold Regression (lightGBM+XG) Jason\Hosuehold regression for v3\develop_02


## Step 1 - What already exists

Models, package code and the production folder. Nothing is created in this step.

In [2]:
def listing(folder, pattern):
    rows = [{"file": p.relative_to(ROOT).as_posix(), "size_kb": round(p.stat().st_size / 1024, 1)}
            for p in sorted(folder.rglob(pattern)) if "__pycache__" not in p.parts]
    return pd.DataFrame(rows)

print("Saved regression models and results:")
display(listing(MODELS, "*.joblib"))
print("Regression package code:")
display(listing(PACKAGE_DIR, "*.py"))
print("Production folder modeling/ (before this notebook writes anything):")
display(listing(MODELING_DIR, "*.py"))

Saved regression models and results:


,file,size_kb
0,models/regression/ensemble_weight_and_margins....,0.2
1,models/regression/final_hyperparameters.joblib,0.4
2,models/regression/lgbm_log_model.joblib,163.8
3,models/regression/xgb_log_model.joblib,41.3


Regression package code:


,file,size_kb
0,my_val_capstone_01/regression/__init__.py,0.2
1,my_val_capstone_01/regression/config.py,0.5
2,my_val_capstone_01/regression/dataset.py,9.7
3,my_val_capstone_01/regression/features.py,14.7
4,my_val_capstone_01/regression/modeling/__init_...,0.0
5,my_val_capstone_01/regression/modeling/predict.py,3.1
6,my_val_capstone_01/regression/modeling/train.py,11.1


Production folder modeling/ (before this notebook writes anything):


,file,size_kb
0,modeling/predict.py,4.2
1,modeling/reg_predict_pipeline.py,4.3


## Step 2 - Functions that can be reused

The functions are read from the `.py` files with `ast`, so the list is always the current code. The pipeline **imports** these and does not copy them, which keeps one source of truth.

In [3]:
def functions_in(path):
    tree = ast.parse(path.read_text(encoding="utf8"))
    out = []
    for node in tree.body:
        if isinstance(node, ast.FunctionDef):
            args = ", ".join(a.arg for a in node.args.args)
            doc = (ast.get_docstring(node) or "").split("\n")[0]
            out.append({"function": f"{node.name}({args})", "what it does": doc})
    return pd.DataFrame(out)

for name in ["features.py", "modeling/predict.py"]:
    print(f"my_val_capstone_01/regression/{name}")
    display(functions_in(PACKAGE_DIR / name))

my_val_capstone_01/regression/features.py


,function,what it does
0,category_column(category),Name of the feature that holds the value docum...
1,"featurize_sample(sample_items, prop_row, conte...",Turn the N documented items of one home into o...
2,floor_to_checkpoint(n_items),"Largest basket size (10, 20, 30 or 50) reached..."
3,"basket_features(basket, context, feature_cols,...",Regression features of the earliest-N basket o...
4,load_regression_models(models_dir),Load the two saved regression models and the f...
5,"build_regression_state_features(model_state, a...",Add the five regression columns to the classif...
6,"main(model_state_path, xlsx_path, models_dir, ...",Optional: write model_state.csv plus the five ...


my_val_capstone_01/regression/modeling/predict.py


,function,what it does
0,_load_artefacts(),
1,checkpoint_for(n_documented_items),"Largest basket size (10, 20, 30 or 50) that th..."
2,predict_household_value(feature_row),Estimate the total contents value of a household.


**What is reused**

- `basket_features(basket, context, feature_cols, contents_categories)` turns the documented items of one home into one row of model features, including the `property_type_*` and `occupancy_type_*` one-hot columns.
- `floor_to_checkpoint(n)` picks the largest basket size (10, 20, 30 or 50) that the documented items reach.
- `load_regression_models()` loads the two models and the frozen blend weight and margins.
- `predict_household_value(feature_row)` blends the two models and returns the estimate and the 70% interval.

**What is missing, and what the pipeline adds:** a step that accepts the **raw documented items and the property details**, checks them, selects the basket and calls the functions above.

## Step 3 - What the saved models need

The exact feature columns (and their order), the blend weight and the margins come from the saved files, so the pipeline never hard-codes them.

In [4]:
xgb_model = joblib.load(MODELS / "xgb_log_model.joblib")
lgbm_model = joblib.load(MODELS / "lgbm_log_model.joblib")
frozen = pd.Series(joblib.load(MODELS / "ensemble_weight_and_margins.joblib"))

FEATURES = list(xgb_model.feature_names_in_)
assert FEATURES == list(lgbm_model.feature_name_), "the two models must use the same columns in the same order"
print(f"{len(FEATURES)} feature columns:", FEATURES)
print()
print("blend weight (XGBoost):", frozen["blend_weight_xgboost"], "| LightGBM:", round(1 - frozen["blend_weight_xgboost"], 2))
print("confidence level:", frozen["confidence_level"])
print("margin per basket size:", {k: round(v, 4) for k, v in frozen.items() if k.startswith("margin_N=")})
print("property types the model knows:", [c[len("property_type_"):] for c in FEATURES if c.startswith("property_type_")])
print("occupancy types the model knows:", [c[len("occupancy_type_"):] for c in FEATURES if c.startswith("occupancy_type_")])

assets = pd.read_parquet(INTERIM / "assets.parquet")
CONTENTS_CATEGORIES = sorted(assets.loc[assets["Category"] != "Vehicles", "Category"].unique())
print("contents categories (Vehicles excluded):", CONTENTS_CATEGORIES)

26 feature columns: [np.str_('item_count'), np.str_('sum_value'), np.str_('mean_value'), np.str_('median_value'), np.str_('max_value'), np.str_('min_value'), np.str_('std_value'), np.str_('n_categories_seen'), np.str_('cat_appliances'), np.str_('cat_clothings'), np.str_('cat_electronics'), np.str_('cat_furniture'), np.str_('cat_jewellery_and_personal_items'), np.str_('cat_other_items'), np.str_('number_of_occupants'), np.str_('number_of_bedrooms'), np.str_('property_valuation_aud'), np.str_('n_documented'), np.str_('property_type_apartment'), np.str_('property_type_duplex'), np.str_('property_type_house'), np.str_('property_type_semi-detached'), np.str_('property_type_terrace'), np.str_('property_type_villa'), np.str_('occupancy_type_owner'), np.str_('occupancy_type_tenant')]

blend weight (XGBoost): 0.28 | LightGBM: 0.72
confidence level: 0.7
margin per basket size: {'margin_N=10': 0.8278, 'margin_N=20': 0.5111, 'margin_N=30': 0.4258, 'margin_N=50': 0.5903}
property types the model kn

## Step 4 - Design of the pipeline

```
 INPUT
 |-- items: list of {Category, Current_Estimated_Value}   (the customer's documented items)
 `-- property: {Property_Type, Occupancy_Type, Number_Of_Occupants,
                Number_Of_Bedrooms, Property_Valuation_AUD}
        |
        v
 1 VALIDATE   drop Vehicles and items without a value; fewer than 10 items -> clear error;
              unknown property or occupancy type -> clear error
        |
        v
 2 CHECKPOINT floor_to_checkpoint(n) -> 10 / 20 / 30 / 50   (e.g. 49 items -> basket of 30)
        |        the FIRST N items in the order given form the basket (deterministic)
        v
 3 FEATURES   basket_features(...) -> 26 columns in the saved order
        |
        v
 4 PREDICT    predict_household_value(row): expm1(XGBoost), expm1(LightGBM),
        |     blend with the frozen weight
        v
 5 INTERVAL   estimate x (1 +/- margin of N); lower bound never below the documented value
        |
        v
 OUTPUT dict: estimate, xgboost / lightgbm estimates, lower, lower_floored, upper, margin, n_documented
```

Two entry points are written: `predict_from_items(items, property_info)` for raw input and `predict_from_features(feature_row)` for a row that is already built (used for the check against the saved test predictions).

## Step 5 - Name of the production file

Change `PIPELINE_FILENAME` if another name is wanted. The file is written to `modeling/`, next to the classification `predict.py`.

In [5]:
PIPELINE_FILENAME = "reg_predict_pipeline.py"       # <- change the name here
PIPELINE_PATH = MODELING_DIR / PIPELINE_FILENAME
print("the pipeline will be written to:", PIPELINE_PATH)
print("a file with this name already exists:", PIPELINE_PATH.exists())

the pipeline will be written to: D:\UTS MDSI\36127\Project\Client dataset\Notebook\Hosuehold Regression (lightGBM+XG) Jason\Hosuehold regression for v3\develop_02\modeling\reg_predict_pipeline.py
a file with this name already exists: True


## Step 6 - Write the pipeline file

The text of the file is kept in `PIPELINE_SOURCE` and written to `PIPELINE_PATH`. Running this cell again overwrites only this one file.

In [6]:
PIPELINE_SOURCE = r'''"""Household contents-value regression: production prediction pipeline.

Raw documented items and property details in, estimate and 70% interval of the total contents value out.
The models, blend weight and margins are read from models/regression/ and are never changed here.
Written by notebooks/regression/08_ky_reg_prediction_pipeline.ipynb.
"""

import sys
from functools import lru_cache
from pathlib import Path

import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from my_val_capstone_01.regression.config import MODELS_DIR  # noqa: E402
from my_val_capstone_01.regression.features import (  # noqa: E402
    CHECKPOINTS,
    basket_features,
    floor_to_checkpoint,
    load_regression_models,
)
from my_val_capstone_01.regression.modeling.predict import predict_household_value  # noqa: E402

CONTENTS_CATEGORIES = [
    "Appliances",
    "Clothings",
    "Electronics",
    "Furniture",
    "Jewellery and Personal Items",
    "Other Items",
]
EXCLUDED_CATEGORIES = ["Vehicles"]
ITEM_FIELDS = ["Category", "Current_Estimated_Value"]
PROPERTY_FIELDS = [
    "Property_Type",
    "Occupancy_Type",
    "Number_Of_Occupants",
    "Number_Of_Bedrooms",
    "Property_Valuation_AUD",
]


@lru_cache(maxsize=1)
def _feature_columns() -> tuple:
    xgb_model, _, _ = load_regression_models(MODELS_DIR)
    return tuple(xgb_model.feature_names_in_)


def _known_levels(prefix: str) -> list:
    return [c[len(prefix):] for c in _feature_columns() if c.startswith(prefix)]


def predict_from_features(feature_row: pd.DataFrame) -> dict:
    """Estimate from one row of model features (see my_val_capstone_01.regression.features)."""
    return predict_household_value(feature_row)


def predict_from_items(items, property_info: dict) -> dict:
    """
    Estimate the total contents value of a household from its documented items.

    Parameters
    ----------
    items : list of dict or pd.DataFrame
        One entry per documented item with ``Category`` and ``Current_Estimated_Value``.
        The first N items (N = 10, 20, 30 or 50, the largest size reached) form the basket.
    property_info : dict
        ``Property_Type``, ``Occupancy_Type``, ``Number_Of_Occupants``,
        ``Number_Of_Bedrooms`` and ``Property_Valuation_AUD``.

    Returns
    -------
    dict
        Estimate, the two model estimates and the 70% interval, all in dollars.
    """
    items_df = pd.DataFrame(items)
    missing = [c for c in ITEM_FIELDS if c not in items_df.columns]
    if missing:
        raise ValueError(f"items are missing the fields {missing}.")
    missing = [f for f in PROPERTY_FIELDS if f not in property_info]
    if missing:
        raise ValueError(f"property_info is missing the fields {missing}.")

    # 1 validate: vehicles are not contents, and an item without a value cannot be summed
    items_df = items_df[~items_df["Category"].isin(EXCLUDED_CATEGORIES)]
    items_df = items_df[items_df["Current_Estimated_Value"].notna()].reset_index(drop=True)
    for field, prefix in [("Property_Type", "property_type_"), ("Occupancy_Type", "occupancy_type_")]:
        if property_info[field] not in _known_levels(prefix):
            raise ValueError(f"{field} must be one of {_known_levels(prefix)}, got {property_info[field]!r}.")

    # 2 checkpoint
    n_documented = floor_to_checkpoint(len(items_df))
    if n_documented == 0:
        raise ValueError(f"at least {CHECKPOINTS[0]} valued contents items are needed, got {len(items_df)}.")
    basket = items_df.iloc[:n_documented]

    # 3 features, in the column order of the saved models
    feats = basket_features(basket, pd.Series(property_info), list(_feature_columns()), CONTENTS_CATEGORIES)
    feature_row = pd.DataFrame([feats])[list(_feature_columns())]

    # 4 and 5 predict, blend and interval
    return predict_household_value(feature_row)


if __name__ == "__main__":
    demo_items = [{"Category": CONTENTS_CATEGORIES[i % 6], "Current_Estimated_Value": 500.0 + 150 * i} for i in range(20)]
    demo_property = {
        "Property_Type": "house",
        "Occupancy_Type": "owner",
        "Number_Of_Occupants": 3,
        "Number_Of_Bedrooms": 3,
        "Property_Valuation_AUD": 900000,
    }
    print(predict_from_items(demo_items, demo_property))
'''

MODELING_DIR.mkdir(exist_ok=True)
PIPELINE_PATH.write_text(PIPELINE_SOURCE, encoding="utf8")
print(f"written: {PIPELINE_PATH.relative_to(ROOT)}  ({len(PIPELINE_SOURCE.splitlines())} lines)")

written: modeling\reg_predict_pipeline.py  (117 lines)


## Step 7 - Check the saved file

The file is imported **by the name set in Step 5**, so the check always tests the file that was just written.

1. **Same numbers as the saved test predictions:** every test row goes through `predict_from_features` and is compared with `test_ensemble_predictions.parquet` (written by `05_ky_test_evaluation`).
2. **Raw items give the same result as the training code:** for test homes, the first N items are sent through `predict_from_items`, and the result is compared with the one built from the training function `featurize_sample` plus one-hot columns.
3. **Error handling:** fewer than 10 items, an unknown property type, a missing field, and a basket between two checkpoints.

In [7]:
sys.path.insert(0, str(MODELING_DIR))
importlib.invalidate_caches()
pipeline = importlib.import_module(PIPELINE_PATH.stem)
importlib.reload(pipeline)
print("imported:", pipeline.__file__)

# 1 parity with the saved test predictions
test_df = pd.read_parquet(PROCESSED / "test_df.parquet")
saved = pd.read_parquet(INTERIM / "test_ensemble_predictions.parquet")
res = pd.DataFrame([pipeline.predict_from_features(test_df.iloc[[i]]) for i in range(len(test_df))])
d_est = (res["estimate"] - saved["ens"]).abs().max()
d_up = (res["upper"] - saved["upper"]).abs().max()
d_lo = (res["lower_floored"] - saved["lower_floor"]).abs().max()
print(f"1. test rows: {len(res)} | max |estimate - saved| = {d_est:.4f} | upper = {d_up:.4f} | floored lower = {d_lo:.4f}")
assert d_est < 0.01 and d_up < 0.01 and d_lo < 0.01

2026-10-06 14:55:48.542 | INFO     | my_val_capstone_01.config:<module>:11 - PROJ_ROOT path is: D:\UTS MDSI\36127\Project\Client dataset\Notebook\Hosuehold Regression (lightGBM+XG) Jason\Hosuehold regression for v3\develop_02


imported: D:\UTS MDSI\36127\Project\Client dataset\Notebook\Hosuehold Regression (lightGBM+XG) Jason\Hosuehold regression for v3\develop_02\modeling\reg_predict_pipeline.py


1. test rows: 150 | max |estimate - saved| = 0.0057 | upper = 0.0062 | floored lower = 0.0050


In [8]:
# 2 raw items -> features -> prediction, against the training code path
from my_val_capstone_01.regression.features import featurize_sample
from my_val_capstone_01.regression.modeling.predict import predict_household_value

properties = pd.read_parquet(INTERIM / "properties.parquet").set_index("Property_ID")
contents = assets[(assets["Category"] != "Vehicles") & assets["Current_Estimated_Value"].notna()]
ITEM_COLS = ["Category", "Current_Estimated_Value"]
PROP_COLS = ["Property_Type", "Occupancy_Type", "Number_Of_Occupants", "Number_Of_Bedrooms", "Property_Valuation_AUD"]

def reference_prediction(basket, prop_row):
    """The training path: featurize_sample + one-hot columns, then the saved models."""
    f = featurize_sample(basket, prop_row, CONTENTS_CATEGORIES)
    f["n_documented"] = len(basket)
    for col in FEATURES:
        if col.startswith("property_type_"):
            f[col] = f["property_type"] == col[len("property_type_"):]
        if col.startswith("occupancy_type_"):
            f[col] = f["occupancy_type"] == col[len("occupancy_type_"):]
    row = pd.DataFrame([f])[FEATURES]
    return predict_household_value(row)

checked = []
for pid in test_df["Property_ID"].unique():
    items = contents[contents["Property_ID"] == pid]
    for n in (10, 20, 30, 50):
        if len(items) < n:
            continue
        basket = items[ITEM_COLS].iloc[:n]
        prop_row = properties.loc[pid, PROP_COLS]
        got = pipeline.predict_from_items(basket.to_dict("records"), prop_row.to_dict())
        want = reference_prediction(basket, prop_row)
        assert got == want, (pid, n, got, want)
        checked.append({"Property_ID": pid, "N": n, "estimate": got["estimate"], "lower": got["lower_floored"], "upper": got["upper"]})
checked = pd.DataFrame(checked)
print(f"2. {len(checked)} baskets from {checked['Property_ID'].nunique()} test homes: identical to the training code path")
checked.head(8)

2. 30 baskets from 13 test homes: identical to the training code path


,Property_ID,N,estimate,lower,upper
0,PRP-000040,10,41666.08,24266.00,76159.22
1,PRP-000083,10,36679.29,9346.00,67044.13
2,PRP-000083,20,34761.15,16995.59,52526.70
3,PRP-000083,30,82054.54,47116.99,116992.09
4,PRP-000083,50,108208.45,83355.00,172084.90
5,PRP-000104,10,45333.41,8080.00,82862.54
6,PRP-000104,20,48278.94,23604.78,72953.10
7,PRP-000104,30,45420.07,26080.91,64759.23


In [9]:
# 3 error handling and checkpoint flooring
prop = properties.loc[test_df["Property_ID"].iloc[0], PROP_COLS].to_dict()
some_items = contents[ITEM_COLS].head(60).to_dict("records")

for label, call in [
    ("fewer than 10 items", lambda: pipeline.predict_from_items(some_items[:9], prop)),
    ("unknown property type", lambda: pipeline.predict_from_items(some_items[:20], {**prop, "Property_Type": "castle"})),
    ("missing property field", lambda: pipeline.predict_from_items(some_items[:20], {k: v for k, v in prop.items() if k != "Property_Type"})),
]:
    try:
        call()
        raise AssertionError(label + " should have been rejected")
    except ValueError as e:
        print(f"3. rejected ({label}): {e}")

r49 = pipeline.predict_from_items(some_items[:49], prop)
r30 = pipeline.predict_from_items(some_items[:30], prop)
assert r49["n_documented"] == 30 and r49 == r30
print("3. 49 items use the basket of 30:", r49["n_documented"])

3. rejected (fewer than 10 items): at least 10 valued contents items are needed, got 9.
3. rejected (unknown property type): Property_Type must be one of ['apartment', 'duplex', 'house', 'semi-detached', 'terrace', 'villa'], got 'castle'.
3. rejected (missing property field): property_info is missing the fields ['Property_Type'].
3. 49 items use the basket of 30: 30


## What this notebook produced

- **One new file:** the pipeline in `modeling/` (`reg_predict_pipeline.py`, or the name set in Step 5). It imports the existing regression code and the saved models, and adds the step from raw items and property details to an estimate with a 70% interval.
- **Nothing else was changed:** no model, table, notebook or package file is written by this notebook.
- **Checks passed:** the pipeline reproduces the saved test predictions, and the raw-items path gives exactly the same result as the training code path.
- **Use:** `from reg_predict_pipeline import predict_from_items` (with `modeling/` on the path), or run `python modeling/reg_predict_pipeline.py` for a demo.